# 04: Turn your calculation into a cloud job

You will write a small Python program, save it where Jiuding can read it, submit it, and retrieve its result.
You need a configured Jiuding development environment with shared storage. See [Setup](../../SETUP.md).
The remote cells do nothing until you explicitly enable submission.

## 1. Choose where the program and results will live

The folder below must be visible inside your job container as well as this notebook.
This is why we use the shared project rather than a file stored only on a laptop.

In [ ]:
from pathlib import Path

# Find the checkout whether Jupyter started in the repository or this folder.
ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "flagquantum").is_dir()
)
WORKSHOP = ROOT / "workshops/flagos2026"
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(exist_ok=True)
print("Your result folder:", OUTPUTS)


## 2. Write the program

This is the Bell calculation you built in notebook 01, now inside a `main()` function.
Jiuding's worker calls `main()` and saves its return value. The function can call other functions and import your shared modules.
For this example, a list of probabilities is easy to save as JSON.

In [ ]:
program_text = """import flagquantum as fq


def main():
    circuit = fq.Circuit(2)
    circuit.h(0)
    circuit.cx(0, 1)
    options = fq.ExecutionOptions(device="cpu", mode="statevector")
    result = fq.run(circuit, options=options)
    state = result.to_statevector().reshape(-1)
    return {
        "device": str(state.device),
        "probabilities": state.abs().square().tolist(),
    }
"""
script_path = OUTPUTS / "my_bell_job.py"
script_path.write_text(program_text)
print("Saved program:", script_path)


## 3. Try your program locally first

`runpy.run_path` loads the file we just wrote. Calling its `main` function lets us catch basic mistakes before using a cloud queue.

In [ ]:
import runpy

program = runpy.run_path(str(script_path))
local_result = program["main"]()
print(local_result)


## 4. Describe the cloud environment

An image contains the software installed in the job container. Ask your instructor for its complete address and Python path.
`JiudingClient` handles authentication and job requests. It does not run a job just because you create it.

In [ ]:
import sys
from flagquantum.remote.compute.jiuding import JiudingClient

client = JiudingClient()
image = ""  # Full image address supplied by your instructor.
job_python = sys.executable  # Must also exist inside that image.
submit_job = False
receipt_path = OUTPUTS / "my_bell_job.receipt.json"


## 5. Submit once and save the receipt

A receipt is a JSON file identifying your submission. Keep it so you can check the job after closing this notebook.
Set `submit_job` to `True` in the previous cell when ready. The receipt path must be new; reusing it is rejected to avoid accidentally submitting twice.

In [ ]:
if submit_job:
    if not image:
        raise ValueError("Enter the instructor's image address first.")
    receipt = client.submit(
        script_path,
        image=image,
        python=job_python,
        pythonpath=ROOT,
        receipt=receipt_path,
        cpus=2,
        memory_gib=2,
        gpus=0,
    )
    print(receipt)
else:
    print("No cloud job submitted. The local program is ready.")


## 6. Check progress and read the result

Enable `check_job` after submitting. Loading the saved receipt works even after a kernel restart.
A timeout means we stopped waiting; it does not mean the job stopped.

In [ ]:
import json

check_job = False
if check_job:
    receipt = json.loads(receipt_path.read_text())
    print(client.status(receipt))
    try:
        cloud_result = client.result(receipt, timeout=180)
        print(cloud_result)
        import torch

        torch.testing.assert_close(
            torch.tensor(cloud_result["probabilities"]),
            torch.tensor(local_result["probabilities"]),
        )
    except TimeoutError:
        print("Still waiting. Keep the receipt and check status again later.")


## 7. Stop a job you no longer need

A cancellation request may take time to complete. Check status again until the platform confirms that the job has stopped.

In [ ]:
cancel_job = False
if cancel_job:
    receipt = json.loads(receipt_path.read_text())
    client.cancel(receipt)
    print(client.status(receipt))


## Try a change

Replace the contents of `program_text` with a small training program based on notebook 03.
Return a dictionary containing its loss history. Save it under a new filename and use a new receipt path for this distinct experiment.

For GPU execution, both request a GPU and select CUDA inside the program.
The standalone `scripts/jiuding_job.py` includes that option for the existing GPU Bell example; see its `--help` output.